In [0]:
%sql
use catalog investment_pyspark;

In [0]:
df_bronze = spark.read.table("investment_pyspark.bronze.dividends_raw")
df_bronze.show()

In [0]:
df_bronze.printSchema()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType,IntegerType

In [0]:
df_silver = (df_bronze
             .withColumn("Symbol",F.upper(F.trim(F.col("Symbol"))))
             .withColumn("Symbol",F.regexp_replace(F.col("Symbol"),"[^A-Za-z0-9]+",""))
             .withColumn("Symbol",F.regexp_replace(F.col("Symbol"),"[0-9]+$",""))
             .withColumn("Ex_date",F.to_date(F.col("Ex_date"),"yyyy-MM-dd"))
             .withColumn("Qty",F.col("Qty").cast(IntegerType()))
             .withColumn("Dividend_per_share",F.col("Dividend_per_share").cast(DecimalType(10,2)))
             .withColumn("Total_dividend",F.col("Total_dividend").cast(DecimalType(10,2)))
             .withColumn("Silver_timestamp",F.current_timestamp())
             .filter(
                 F.col("Symbol").isNotNull() 
                 & F.col("Ex_date").isNotNull()
                 & (F.col("Qty")>0)
                 & (F.col("Dividend_per_share")>=0)
                 & (F.col("Total_dividend")>=0)
             )
             .select("Symbol","Ex_date","Qty","Dividend_per_share","Total_dividend","Silver_timestamp")
)


In [0]:
df_silver.printSchema()

In [0]:
df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "True").saveAsTable("investment_pyspark.silver.dividens_cleaned")

In [0]:
%sql
select * from investment_pyspark.silver.dividens_cleaned;